# 1. Klasse · Bildmodell trainieren und fair testen
**DL 5–7 · gemeinsames Projekt-Notebook**

Wir trainieren einen **echten, einfachen Bildklassifikator** auf kleinen 8×8-Pixelbildern. Die Bilder werden im Notebook erzeugt. Ihr müsst keine fremden Bilder hochladen. Das Modell lernt aus Beispielen und wird anschliessend mit **neuen** Bildern getestet. Diese vereinfachten Pixelmuster sind kein Fotoerkennungsmodell.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(12)

def bild(klasse, rausch=0.12):
    x = np.zeros((8,8),dtype=float)
    if klasse == 0: x[:,3:5] = 1.0  # senkrechter Balken
    else: x[3:5,:] = 1.0          # waagrechter Balken
    return np.clip(x+rng.normal(0,rausch,(8,8)),0,1)

def daten(n,rausch=0.12):
    X=np.array([bild(k,rausch).ravel() for k in [0,1] for _ in range(n)])
    y=np.array([k for k in [0,1] for _ in range(n)])
    return X,y
X_train,y_train=daten(12)
X_test,y_test=daten(12) # neue, nicht fürs Training verwendete Bilder
fig,ax=plt.subplots(1,4,figsize=(7,2))
for i,a in enumerate(ax):
    a.imshow(X_train[[0,1,12,13][i]].reshape(8,8),cmap='gray',vmin=0,vmax=1)
    a.set_axis_off()
plt.show()

## DL 5 · Erstes Modell
Ein einfaches logistisches Neuron erhält 64 Pixelwerte. Es lernt Gewichte aus Trainingsbildern. **Vermutet:** Kann es die zwei Balkenrichtungen unterscheiden?

In [ ]:
def trainiere(X,y,runden=300,lernschritt=0.4):
    w=np.zeros(X.shape[1]); b=0.0
    for _ in range(runden):
        z=np.clip(X@w+b,-30,30)
        p=1/(1+np.exp(-z))
        w-=lernschritt*(X.T@(p-y))/len(y)
        b-=lernschritt*np.mean(p-y)
    return w,b

def vorhersagen(X,modell):
    w,b=modell
    return (X@w+b>=0).astype(int)

modell_a=trainiere(X_train,y_train)
print('Trainingsbilder:',len(y_train))
print('Neue Testbilder:',len(y_test))
print('Richtige Testvorhersagen:',sum(vorhersagen(X_test,modell_a)==y_test),'von',len(y_test))

## DL 6 · Fairen Vergleich planen
Entscheidet euch für **eine** Veränderung: mehr Trainingsbilder oder stärkere Bildstörungen. Testet beide Modelle anschliessend auf **denselben** Testbildern. Schreibt eure Hypothese auf der Kursseite auf.

In [ ]:
# Verändert nur EINE der beiden folgenden Einstellungen.
anzahl_bilder = 12  # Versuch: 4 oder 24
bildrauschen = 0.12  # Versuch: 0.35
X_var,y_var=daten(anzahl_bilder,bildrauschen)
modell_b=trainiere(X_var,y_var)
print('Variante B mit',len(y_var),'Trainingsbildern')

## DL 7 · Ergebnisse vergleichen
Führt die folgende Zelle aus. Vergleicht **für jede Klasse getrennt** und erklärt, was eure Veränderung bewirkt hat. Achtung: Kleine Tests können zufällig gut ausfallen.

In [ ]:
for name,modell in [('A',modell_a),('B',modell_b)]:
    p=vorhersagen(X_test,modell)
    print('Modell',name)
    for klasse in [0,1]:
        maske=y_test==klasse
        print(' Klasse',klasse,':',int(np.sum(p[maske]==y_test[maske])),'von',int(np.sum(maske)),'richtig')

fig,ax=plt.subplots(2,5,figsize=(9,4))
for i,a in enumerate(ax.flat):
    a.imshow(X_test[i].reshape(8,8),cmap='gray',vmin=0,vmax=1)
    a.set_title('Ziel '+str(y_test[i])+' / A '+str(vorhersagen(X_test[i:i+1],modell_a)[0]),fontsize=9)
    a.set_axis_off()
plt.tight_layout();plt.show()

## Fachgespräch vorbereiten
Erklärt, was ein Pixel, ein Trainingsbild und ein Testbild ist. Zeigt zwei Resultate und begründet, weshalb nur **eine** Bedingung verändert werden sollte. Speichert das Notebook mit den ausgeführten Zellen.